In [ ]:
# Standard imports and .env loading
import os
from dotenv import load_dotenv

# Load environment variables from a local .env file (if present)
load_dotenv()

# Retrieve and validate Cohere API key from environment variables
cohere_api_key = os.getenv("COHERE_API_KEY")
if not cohere_api_key:
    raise ValueError("COHERE_API_KEY not found in .env file or environment")


In [ ]:
# Import Cohere chat wrapper and any tools we may use
from langchain_cohere import ChatCohere
from langchain_core.tools import tool
import requests  # used by some tools/examples


In [ ]:
## Tool -1

In [ ]:
# Simple web search tool from the community integrations
from langchain_community.tools import DuckDuckGoSearchRun

# Instantiate the search tool and run a quick example query
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke("what is the capital of France")


In [ ]:
## Tool -2

In [ ]:
@tool
def get_weather_data(location: str) -> str:
    """Get the current weather data for a given location."""
    url = "api endpoint"
    response = requests.get(url)
    return response.json()

In [ ]:
# Display the raw search results object (for exploration)
results


'By the end of the 12th century, Paris had become the political, economic, religious, and cultural capital of France .[22] Maurice de Sully, bishop of Paris, started the construction of the Notre Dame Cathedral in 1163, and was completed after 182 years.[23] After the marshland between the river... The capital city of France is Paris. Paris is not only the political and administrative center of France but is also renowned for its rich history, culture, art, fashion, and architecture. It is famously known as " The City of Light" (La Ville Lumière) and is one of the most visited cities in the world. Where in the World is Paris found? Paris is the capital of France (French Republic), situated in the Western Europe subregion of Europe. In Paris, the currency used is Euro (€), which is the official currency used in France . The Latitude, Longitude cordinates of Paris are 48.8534, 2.3488. Learn everything about the capital of France , Paris. Explore its history, geography, landmarks, culture

In [ ]:
# Initialize the Cohere chat LLM wrapper with the API key
llm = ChatCohere(
    cohere_api_key=cohere_api_key,
    model="command-a-03-2025",  # model selection can be changed
)


In [ ]:
# Query the LLM directly (simple invocation example)
llm.invoke("what is the capital of France")


AIMessage(content="The capital of France is **Paris**. It is the country's largest city and main cultural and commercial center, located on the River Seine in northern France at the heart of the Île-de-France region.", additional_kwargs={'id': '5b363fe2-db60-434d-8e58-ad1a13007363', 'finish_reason': 'COMPLETE', 'content': "The capital of France is **Paris**. It is the country's largest city and main cultural and commercial center, located on the River Seine in northern France at the heart of the Île-de-France region.", 'token_count': {'input_tokens': 501.0, 'output_tokens': 44.0}}, response_metadata={'id': '5b363fe2-db60-434d-8e58-ad1a13007363', 'finish_reason': 'COMPLETE', 'content': "The capital of France is **Paris**. It is the country's largest city and main cultural and commercial center, located on the River Seine in northern France at the heart of the Île-de-France region.", 'token_count': {'input_tokens': 501.0, 'output_tokens': 44.0}}, id='run--361b9f95-7b69-462d-acf6-d8ff5104

In [ ]:
# Agent utilities from LangChain
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub


In [ ]:
# Step 2 - pull the reAct agent prompt template from the LangChain hub
prompt = hub.pull("hwchase17/react")


In [ ]:
# Inspect the pulled prompt/template
prompt


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [ ]:
# Step 3 - create the reAct-style agent using the prompt and tools

In [ ]:
agent = create_react_agent(
    llm=llm,
    prompt=prompt,  # use the prompt template pulled from the hub
    tools=[search_tool, get_weather_data],  # provide tools the agent can call
)


In [ ]:
# Step 4 - wrap the agent with an executor which handles tool routing and runs


In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],  # the executor needs access to the tools too
    verbose=True,  # enable verbose logging for debugging
)


In [ ]:
# Run the agent executor with a simple input and print the result
result = agent_executor.invoke({"input": "What’s the current weather in New York"})
print(result)




> Entering new AgentExecutor chain...
Question: what is the capital of France  
Thought: I know that the capital of France is a well-known fact and does not require a search for current events. However, to follow the format and ensure accuracy, I will use the search tool to confirm the information.  
Action: duckduckgo_search  
Action Input: capital of FranceO3 days ago - Paris is the capital and largest city of France, with an estimated city population of 2,048,472 in an area of 105.4 km2 (40.7 sq mi), and a metropolitan population of 13,171,056 as of January 2025. Located on the river Seine in the centre of the Île-de-France region, it is the largest metropolitan ... June 23, 2025 - This is a chronological list of capitals of France. The capital of France has been Paris since its liberation in 1944. ... Paris (987–1419), the residence of the Kings of France, although they were consecrated at Reims. Orléans (1108), one of the few consecrations of a French monarch to ... 2 days ago -